[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Saving and Loading Boosters


## What you will be able to do

Save a trained LightGBM or XGBoost model to a file, load it back, check that it predicts exactly as
it did, and know what each library's file keeps and what it leaves behind.

## The idea

### The problem

A model of the day's lowest temperature is trained once, on a few years of station-days, and used
every morning for a forecast. The process that trains it ends, and the one that forecasts starts
fresh, on another day and perhaps on another machine. Training again each morning would waste the
time, and could give a slightly different model if the data had changed. The trained trees have to be
written down and read back.

Python's `pickle`, which `joblib` builds on, can write the whole object, the scikit-learn class and
all, as the **scikit-learn, Deep Dive** guide showed. A pickle can only be read by Python, with the same
classes available, and that guide met the `InconsistentVersionWarning` scikit-learn gives when a pickle
is loaded under another version. Both boosting libraries write files of their own instead, in a format
the library reads back by itself.

### A formal definition

> A **model file** stores a booster's trees, the settings that decide how it turns them into a
> prediction, and the names and types of the features it expects, in a format the library can read
> without the Python object that wrote it. LightGBM writes **text**. XGBoost writes **JSON**, which is
> text, or **UBJSON**, a binary encoding of the same structure, chosen by the file's extension.

### Why it works that way

A tree is a list of splits and leaf values, so a model is a long list of numbers that any program can
read. LightGBM's file is plain text: a header with the feature names, then every tree, then the
settings it was trained with. XGBoost's JSON holds the trees, the objective and the feature names as
nested objects, and UBJSON stores the same structure in fewer bytes. Neither holds the settings XGBoost
was trained with, which LightGBM's text file lists near its end.

What the file keeps is what the booster needs to predict, and LightGBM's file adds its training
settings. The scikit-learn class around the booster holds more, and some of that does not travel. A model trained with early
stopping is where this matters. LightGBM keeps only the trees up to the best round in the booster
itself, so its file holds those. XGBoost keeps every round it trained and writes the best round down
as an attribute beside them, and only the scikit-learn class reads that attribute back when it
predicts.

The test of a saved model is that the loaded copy gives the same predictions as the original, value
for value. Loading without an error comes first, and the comparison of predictions is the test.

### Where this shows up

The **Early Stopping** notebook showed that LightGBM drops the trees after its best round while
XGBoost keeps them, which decides what each file holds here, and the **A Frost Forecast** notebook
saves both of its classifiers and loads them to forecast a planned night.

### What this notebook covers

- LightGBM's text file, read line by line, and the booster loaded from it
- XGBoost's JSON and UBJSON files, and what the JSON holds
- the check that a loaded model predicts what the original did
- what a file keeps after early stopping, and what the scikit-learn class leaves behind
- four mistakes in saving and loading

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import lightgbm as lgb
import pandas as pd

x = pd.DataFrame({"a": [0, 1, 2, 3] * 25})
model = lgb.LGBMRegressor(n_estimators=10, min_child_samples=5, verbose=-1).fit(x, 2 * x["a"])
model.booster_.save_model("model.txt")
loaded = lgb.Booster(model_file="model.txt")
print(loaded.num_trees(), loaded.predict(x[:4]).round(3))
print(model.predict(x[:4]).round(3))
```

```
10 [1.046 2.349 3.651 4.954]
[1.046 2.349 3.651 4.954]
```

The model is saved through `booster_`, the trees inside the scikit-learn class, and the file loads as
a `Booster` with the same 10 trees and the same predictions.

## Setup

The standard library's tools for files, the two boosting libraries, and the table.

- `Path` names the files, `json` reads XGBoost's JSON file, and `shutil` removes the folder at the end
- `warnings` catches the warning XGBoost gives in the Common errors
- `numpy` and `pandas` build the table and compare predictions
- `lightgbm` and `xgboost` are LightGBM and XGBoost, and the cell prints their versions

`station_days` makes the same 20,000 made-up station-days as the rest of the guide, from a fixed
seed, so the table is the same on every run, and the cell splits the stations into 300 for training,
50 for validation and 50 for testing. The models train on the training stations and stop early on the
validation stations, and their predictions for the test stations are compared before and after
saving. Every file goes in `scratch`, which the cell makes and the notebook's last cell removes.

In [1]:
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

scratch = Path("scratch")
scratch.mkdir(exist_ok=True)
print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "| files go in", scratch)

lightgbm 4.6.0 | xgboost 3.4.1 | files go in scratch


## Worked examples

### The models to save

Both models train with early stopping on the validation stations, so each has a best round, and the
two libraries treat it differently.

In [2]:
lightgbm_model = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.1, random_state=0, verbose=-1)
lightgbm_model.fit(train[FEATURES], train["tmin"], eval_set=[(valid[FEATURES], valid["tmin"])],
                   callbacks=[lgb.early_stopping(50, verbose=False)])
print("LightGBM best round", lightgbm_model.best_iteration_, "| trees in its booster", lightgbm_model.booster_.num_trees())

xgboost_model = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.1, early_stopping_rounds=50, random_state=0)
xgboost_model.fit(train[FEATURES], train["tmin"], eval_set=[(valid[FEATURES], valid["tmin"])], verbose=False)
print("XGBoost best round", xgboost_model.best_iteration, "| rounds in its booster",
      xgboost_model.get_booster().num_boosted_rounds())

LightGBM best round 120 | trees in its booster 120
XGBoost best round 76 | rounds in its booster 127


LightGBM's best round is 120, and its booster holds 120 trees, so the trees trained after the best
round are already gone. XGBoost counts rounds from 0, so its best round, 76, is the 77th, and its
booster holds those 77 rounds and the 50 that followed without improving, 127 in all.

### LightGBM's text file

`save_model` belongs to the booster, so the scikit-learn class reaches it through `booster_`. The file
is plain text, and its first lines name the features in order.

In [3]:
lightgbm_file = scratch / "lightgbm_model.txt"
lightgbm_model.booster_.save_model(lightgbm_file)

lines = lightgbm_file.read_text().splitlines()
for line in lines:
    if line.startswith(("version=", "objective=", "feature_names=")):
        print(line)
print("trees in the file:", sum(line.startswith("Tree=") for line in lines))
print([line for line in lines if line.startswith("pandas_categorical")])

version=v4
objective=regression
feature_names=latitude elevation_m coast_km terrain day_of_year cloud_cover wind_ms snow_cm
trees in the file: 120
['pandas_categorical:[["coastal", "fjord", "inland", "valley", "mountain"]]']


The file names its format's version and its objective, then the eight features in the order the model
was given them, and holds 120 trees. Its last line records `terrain`'s categories in order, which is
how a loaded booster turns a category's name into the number it was trained on.

In [4]:
lightgbm_loaded = lgb.Booster(model_file=lightgbm_file)
print(type(lightgbm_loaded).__name__, "with", lightgbm_loaded.num_trees(), "trees")
print("same predictions:", np.array_equal(lightgbm_loaded.predict(test[FEATURES]), lightgbm_model.predict(test[FEATURES])))

Booster with 120 trees
same predictions: True


The file loads as a `Booster`, the trees without the scikit-learn class around them, and gives the
same prediction for every test day.

### XGBoost's JSON and UBJSON files

XGBoost's `save_model` is on the scikit-learn class itself, and the extension of the file name
chooses the format.

In [5]:
json_file, ubj_file = scratch / "xgboost_model.json", scratch / "xgboost_model.ubj"
xgboost_model.save_model(json_file)
xgboost_model.save_model(ubj_file)
print("UBJSON is the smaller file:", ubj_file.stat().st_size < json_file.stat().st_size)

learner = json.loads(json_file.read_text())["learner"]
print(learner["feature_names"])
print(learner["feature_types"])
print(learner["attributes"]["best_iteration"])

UBJSON is the smaller file: True
['latitude', 'elevation_m', 'coast_km', 'terrain', 'day_of_year', 'cloud_cover', 'wind_ms', 'snow_cm']
['float', 'int', 'float', 'c', 'int', 'float', 'float', 'float']
76


The UBJSON file is the smaller of the two. The JSON names the eight features with a type for each,
`c` marking the category, and keeps the best round, 76, among its attributes.

In [6]:
xgboost_loaded = xgb.XGBRegressor()
xgboost_loaded.load_model(ubj_file)
print("best round:", xgboost_loaded.best_iteration)
print("same predictions:", np.array_equal(xgboost_loaded.predict(test[FEATURES]), xgboost_model.predict(test[FEATURES])))
print("n_estimators and learning_rate after loading:", xgboost_loaded.get_params()["n_estimators"],
      xgboost_loaded.get_params()["learning_rate"])
print("eta, max_depth or n_estimators in the JSON file:",
      any(f'"{name}"' in json_file.read_text() for name in ("eta", "max_depth", "n_estimators")))

best round: 76
same predictions: True
n_estimators and learning_rate after loading: None None
eta, max_depth or n_estimators in the JSON file: False


Loaded into an `XGBRegressor`, the model reads its best round back from the file and predicts every
test day exactly as the original did. Its `n_estimators` and `learning_rate` come back as `None`.
XGBoost's file holds the trees and what the booster needs to predict with them, the objective among
them, and none of the settings the model was trained with, so a loaded model predicts as it should and
has to be given its settings again before it is refitted. LightGBM's text file does list its training
settings, as Your turn shows.

### Categories that arrive in another order

A table built on another day can list `terrain`'s categories in another order, and a category is
stored as a number for its place in that list. Both loaded models are given the test stations with the
order reversed.

In [7]:
reordered = test[FEATURES].copy()
reordered["terrain"] = reordered["terrain"].cat.reorder_categories(list(reversed(TERRAINS)))
print(list(reordered["terrain"].cat.categories))
print("LightGBM unchanged:", np.array_equal(lightgbm_loaded.predict(reordered), lightgbm_loaded.predict(test[FEATURES])))
print("XGBoost unchanged:", np.array_equal(xgboost_loaded.predict(reordered), xgboost_loaded.predict(test[FEATURES])))

['mountain', 'valley', 'inland', 'fjord', 'coastal']
LightGBM unchanged: True
XGBoost unchanged: True


Both loaded models give the same predictions with the categories reversed. Each library stores the
categories it was trained with and matches a new table's categories to them by name, so the order a
new table lists them in does not matter.

## Your turn

Four tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/14-saving-and-loading-boosters-solutions.ipynb).

**1.** Save `lightgbm_model`'s booster with `model_to_string()`, load it back with `lgb.Booster(model_str=...)`, and check that it predicts the test stations as the original does.

In [8]:
# your code here

**2.** Print the lines of LightGBM's file that start with `[num_leaves:` and `[learning_rate:`, the settings the file records near its end.

In [9]:
# your code here

**3.** Train an `XGBClassifier` for frost (`tmin < 0`) with 100 trees, save it to `scratch/frost.json`, load it into a new `XGBClassifier`, and check that `predict_proba` gives the same probabilities.

In [10]:
# your code here

**4.** Read `scratch/xgboost_model.json` and print how many trees it holds, from `learner['gradient_booster']['model']['gbtree_model_param']`, beside the best round.

In [11]:
# your code here

## Common errors

### `AttributeError: 'LGBMRegressor' object has no attribute 'save_model'`

In [12]:
lightgbm_model.save_model(scratch / "model.txt")

AttributeError: 'LGBMRegressor' object has no attribute 'save_model'

`save_model` is a method of LightGBM's `Booster`, and `LGBMRegressor` holds its booster in
`booster_`, so the call is `lightgbm_model.booster_.save_model(...)`, as the worked examples make it.
XGBoost's scikit-learn classes do have a `save_model`, which makes this an easy slip when moving
between the two libraries.

### `TypeError: Loading an estimator with different type. Expecting: classifier, got: regressor`

In [13]:
xgb.XGBClassifier().load_model(json_file)

TypeError: Loading an estimator with different type. Expecting: classifier, got: regressor

XGBoost's file records which kind of scikit-learn class saved it, and `load_model` on another kind
raises. A regressor's file loads into an `XGBRegressor`, and a classifier's into an `XGBClassifier`.

### No error, and a loaded booster that predicts with every tree

In [14]:
plain_booster = xgb.Booster()
plain_booster.load_model(json_file)
test_matrix = xgb.DMatrix(test[FEATURES], enable_categorical=True)
every_tree = plain_booster.predict(test_matrix)
print("rounds in the booster:", plain_booster.num_boosted_rounds(), "| best round:", plain_booster.attr("best_iteration"))
print("same as the model it was saved from:", np.array_equal(every_tree, xgboost_model.predict(test[FEATURES])))

rounds in the booster: 127 | best round: 76
same as the model it was saved from: False


A plain `Booster` loaded from the file has all 127 rounds, with the best round written beside them as
an attribute, and its `predict` uses every round. Its predictions differ from those of the model that
saved it, which stops at the best round, and nothing warns. Give `predict` an `iteration_range` that
ends one past the best round, since the end of the range is left out.

In [15]:
best_round = int(plain_booster.attr("best_iteration"))
up_to_best = plain_booster.predict(test_matrix, iteration_range=(0, best_round + 1))
print("same with iteration_range:", np.array_equal(up_to_best, xgboost_model.predict(test[FEATURES])))

same with iteration_range: True


With `iteration_range=(0, 77)`, which covers rounds 0 to 76, the booster predicts what the original model did.
LightGBM's booster never meets this, since it kept only the trees up to its best round.

### `UserWarning: Saving model in the UBJSON format as default.`

In [16]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", UserWarning)
    xgboost_model.save_model(scratch / "xgboost_model")
for warning in caught:
    message = str(warning.message)
    print(f"{warning.category.__name__}: {message[message.index('Saving'):]}")
print("same bytes as xgboost_model.ubj:", (scratch / "xgboost_model").read_bytes() == ubj_file.read_bytes())

same bytes as xgboost_model.ubj: True


With no extension to go by, XGBoost writes UBJSON and warns. The file it wrote, `xgboost_model`, holds
the same bytes as `xgboost_model.ubj`, and nothing in its name says so. Name the file with `.json` or
`.ubj`. The last cell removes the folder and every file in it.

In [17]:
shutil.rmtree(scratch)
print("scratch removed:", not scratch.exists())

scratch removed: True


## Recap

- You can save a LightGBM model through its booster as text, and an XGBoost model as JSON or UBJSON,
  naming the format by the extension.
- You can load each file back, LightGBM's into a `Booster` and XGBoost's into a `Booster` or its
  scikit-learn class, and check that the loaded model predicts exactly what the original did.
- You can say what each file keeps after early stopping, and that XGBoost's file leaves out the
  settings the model was trained with, which LightGBM's file lists.
- You can predict from a loaded XGBoost `Booster` with `iteration_range`, up to the best round.

## What is next

The **A Frost Forecast** notebook builds a frost classifier from start to finish in both libraries,
chooses it on the validation stations, measures it once on the test stations, and saves it.

---

&#8592; **Previous:** [Reading What They Learned](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/13-reading-what-they-learned.ipynb)  &nbsp;·&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [A Frost Forecast](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/15-a-frost-forecast.ipynb) &#8594;
